LING 430 Fall 2026 

lecture 06b

in-class activity

#### setup

In [ ]:
import nltk
from math import log10, log, exp
nltk.download("gutenberg", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.corpus import gutenberg

tokens = list(gutenberg.words("austen-emma.txt"))
cfd = nltk.ConditionalFreqDist(nltk.bigrams(tokens))
len(tokens)

#### activity 1

write your three guesses on paper first, then run the cell.

In [ ]:
for word in ["so", "very", "I"]:
    print(word, cfd[word].most_common(3))

#### n-grams

an n-gram is a window of n consecutive words.

In [ ]:
words = "colorless green ideas sleep furiously".split()
list(nltk.bigrams(words))

In [ ]:
list(nltk.ngrams(words, 3))

In [ ]:
# character n-grams: the same idea over letters
["".join(pair) for pair in nltk.bigrams("unnatural")]

#### training

count and normalize: P(w | prev) = C(prev w) / C(prev). the markers make sentence starts and ends into ordinary bigrams.

#### activity 2

compute the six probabilities on the slide by hand, then run this cell to check.

In [ ]:
corpus = "<s> I am Sam </s> <s> Sam I am </s> <s> I do not like green eggs and ham </s>".split()
unigrams = nltk.FreqDist(corpus)
bigrams = nltk.FreqDist(nltk.bigrams(corpus))
for prev, w in [("<s>", "I"), ("<s>", "Sam"), ("I", "am"), ("I", "do"), ("Sam", "</s>"), ("am", "Sam")]:
    print(f"P({w} | {prev}) = {bigrams[(prev, w)]} / {unigrams[prev]} = {bigrams[(prev, w)] / unigrams[prev]:.2f}")

the same recipe on Emma: lowercase, markers on every sentence, two frequency distributions, one division.

In [ ]:
sents = [["<s>"] + [w.lower() for w in s] + ["</s>"] for s in gutenberg.sents("austen-emma.txt")]
train = [w for s in sents for w in s]
unigrams = nltk.FreqDist(train)
bigrams = nltk.FreqDist(nltk.bigrams(train))
len(unigrams), len(bigrams)

In [ ]:
def p(w, prev):
    return bigrams[(prev, w)] / unigrams[prev]

p("was", "she"), p("was", "he")

the probability of a sentence: chain the bigrams, and add the logs so the number stays readable.

In [ ]:
sentence = "<s> she was not afraid . </s>".split()
probs = [p(w, prev) for prev, w in nltk.bigrams(sentence)]
[round(x, 4) for x in probs]

In [ ]:
sum(log10(x) for x in probs)

In [ ]:
# word order matters to a bigram model: a zero bigram gives the whole sentence zero
scrambled = "<s> afraid not was she . </s>".split()
[p(w, prev) for prev, w in nltk.bigrams(scrambled)]

#### zeros and smoothing

add-k: add k to every count, and k times the vocabulary size to every denominator, so nothing is zero.

In [ ]:
V = len(unigrams)

def p_smooth(w, prev, k=0.01):
    return (bigrams[(prev, w)] + k) / (unigrams[prev] + k * V)

p_smooth("not", "afraid"), p_smooth("was", "she")

#### evaluation

train on Emma, test on Persuasion. perplexity is the inverse probability of the test text, per word: lower is better.

In [ ]:
test = [w for s in gutenberg.sents("austen-persuasion.txt") for w in ["<s>"] + [x.lower() for x in s] + ["</s>"]]
# words Emma never saw become an unknown-word symbol
test = [w if w in unigrams else "<unk>" for w in test]
V = len(unigrams) + 1
test.count("<unk>") / len(test)

In [ ]:
def perplexity(k, order):
    total = 0
    n = 0
    for prev, w in nltk.bigrams(test):
        if w == "<s>":
            continue
        if order == 1:
            prob = (unigrams[w] + k) / (unigrams.N() + k * V)
        else:
            prob = (bigrams[(prev, w)] + k) / (unigrams[prev] + k * V)
        total = total + log(prob)
        n = n + 1
    return exp(-total / n)

perplexity(0.01, 1), perplexity(0.01, 2)

In [ ]:
# add-one moves too much mass to the zeros: the bigram model ends up worse than the unigram one
perplexity(1, 1), perplexity(1, 2)

#### answers

In [ ]:
# activity 1: see the first cell; the model answers with counts, you answered with a feeling of confidence, and both are distributions
for word in ["so", "very", "I"]:
    print(word, [(w, round(c / cfd[word].N(), 2)) for w, c in cfd[word].most_common(3)])

In [ ]:
# activity 2
# P(I | <s>) = 2/3, P(Sam | <s>) = 1/3, P(am | I) = 2/3, P(do | I) = 1/3, P(</s> | Sam) = 1/2, P(Sam | am) = 1/2